# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a [Croissant schema](https://github.com/mlcommons/croissant/), available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata (using .metadata attributes)
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")

## 2. Data Overview
List available record sets and their `@id`s, fields, and columns.

In [ ]:
# Get all record sets
record_sets = dataset.metadata.recordSet
print(f"Found {len(record_sets)} record set(s):\n")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet name: {getattr(rs, 'name', 'N/A')}")
    print(f"    @id: {rs['@id']}")
    # Fields for each record set
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"    Fields (@id):")
    for field in fields:
        print(f"      - {field['@id'] if isinstance(field, dict) and '@id' in field else field}")
    columns = getattr(rs, 'column', [])
    if not isinstance(columns, list):
        columns = [columns]
    if columns:
        print(f"    Columns (@id):")
        for col in columns:
            print(f"      - {col['@id'] if isinstance(col, dict) and '@id' in col else col}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** Please confirm available record set `@id`s from the overview above. (Below, the first available record set is selected as an example.)

In [ ]:
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

# Obtain the @id for each record set
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))  # Each record is a dict with field @id keys
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns (field @id): {list(df.columns)}\n")
    print(df.head(2), "\n")

# For the remaining cells, we use the first record_set by default (change as appropriate):
main_record_set = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Process and analyze numeric fields in the data. For demonstration, a numeric field (@id) from the main record set is selected.

In [ ]:
# If no record sets available, skip EDA
if main_record_set is not None:
    df = dataframes[main_record_set]
    print(f"Columns in record set {main_record_set}:")
    pprint(list(df.columns))

    # Guess a numeric field by checking column names for common features
    numeric_candidates = [col for col in df.columns if df[col].dtype in ('float', 'int') or df[col].dropna().apply(lambda x: str(x).replace('.', '', 1).isdigit()).any()]
    if not numeric_candidates:
        # Fallback: try any column with 'count', 'mw', 'percent', 'abundance', 'coverage' in @id
        pattern = ['count', 'mw', 'percent', 'abundance', 'coverage', 'pi']
        numeric_candidates = [col for col in df.columns if any(p in str(col).lower() for p in pattern)]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Try casting to float
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 25%): {filtered_df.shape[0]} records\n")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

        # Show normalization
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field (search for 'description' or 'accession' or first non-numeric)
        group_field_candidates = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            print(f"\nGrouped statistics by {group_field} (mean):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(grouped_df.head())
        else:
            print("No suitable group field found.\n")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No record sets found in the dataset.")

## 5. Visualization
Visualize numeric field distributions with histograms and scatterplots.

> **Note:** If this is a large proteomics dataset, some fields may have many unique values. Adjust the number of bins or sample the data as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set is not None and 'numeric_field' in locals():
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    ax.set_xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    # Try scatter vs. the first group field if possible
    if 'group_field' in locals() and group_field and pd.api.types.is_numeric_dtype(df[numeric_field]) and df[group_field].nunique() < 40:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No main record set or numeric field detected.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, explored its metadata, loaded and inspected records by referencing all entities (record sets, fields, columns) using their Croissant `@id`. We performed initial EDA and visualizations on numeric fields such as peptide counts, molecular weights, or abundance measures (field selection is dataset-driven).

This workflow can be extended to downstream applications such as advanced biomarker discovery, comparative analysis across samples, or preparation for machine learning workflows.